In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Verify input directory structure (optional)
print("Input directories:", os.listdir("../input"))



Input directories: ['sample_submission.csv', 'test.csv.zip', 'train.csv.zip', 'labels.csv.zip', 'train.csv', 'new-york-city-taxi-fare-prediction', 'test.csv', 'labels.csv', 'description.md', 'GCP-Coupons-Instructions.rtf', 'sample_submission.csv.zip']


In [2]:
# Correct paths to the full dataset; load a sample to stay memory‑friendly
train_path = "../input/new-york-city-taxi-fare-prediction/train.csv"
test_path = "../input/new-york-city-taxi-fare-prediction/test.csv"

# Load a subset of the training data (e.g., 1,000,000 rows) for faster execution
train = pd.read_csv(
    train_path,
    nrows=1_000_000,  # adjust if more memory is available
    usecols=[
        "key",
        "fare_amount",
        "pickup_datetime",
        "pickup_longitude",
        "pickup_latitude",
        "dropoff_longitude",
        "dropoff_latitude",
        "passenger_count",
    ],
    dtype={
        "key": "object",
        "fare_amount": np.float32,
        "pickup_longitude": np.float32,
        "pickup_latitude": np.float32,
        "dropoff_longitude": np.float32,
        "dropoff_latitude": np.float32,
        "passenger_count": np.int8,
    },
    parse_dates=["pickup_datetime"],
)

test = pd.read_csv(
    test_path,
    usecols=[
        "key",
        "pickup_datetime",
        "pickup_longitude",
        "pickup_latitude",
        "dropoff_longitude",
        "dropoff_latitude",
        "passenger_count",
    ],
    dtype={
        "key": "object",
        "pickup_longitude": np.float32,
        "pickup_latitude": np.float32,
        "dropoff_longitude": np.float32,
        "dropoff_latitude": np.float32,
        "passenger_count": np.int8,
    },
    parse_dates=["pickup_datetime"],
)

print("Train shape:", train.shape, "Test shape:", test.shape)



Train shape: (1000000, 8) Test shape: (9914, 7)


In [3]:
# Drop any rows with missing values (should be rare)
train.dropna(inplace=True)
test.dropna(inplace=True)




In [4]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Compute Haversine distance (km) between two latitude/longitude points.
    Inputs are in decimal degrees.
    """
    R = 6371.0  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c




In [5]:
# Feature engineering for training data
train["distance"] = haversine_distance(
    train["pickup_latitude"],
    train["pickup_longitude"],
    train["dropoff_latitude"],
    train["dropoff_longitude"],
)

train["day_of_week"] = train["pickup_datetime"].dt.dayofweek
train["month"] = train["pickup_datetime"].dt.month
train["hour"] = train["pickup_datetime"].dt.hour



In [6]:
# Prepare features / target and evaluate with a validation split
X = train[["distance", "day_of_week", "month", "hour"]]
y = train["fare_amount"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred_val = model.predict(X_val)
rmse_val = np.sqrt(mean_squared_error(y_val, y_pred_val))
print("Validation RMSE:", rmse_val)



Validation RMSE: 9.829514030513957


In [7]:
# Retrain on the full training set (still the sampled data)
model.fit(X, y)



LinearRegression()

In [8]:
# Feature engineering for test data (same steps as training)
test["distance"] = haversine_distance(
    test["pickup_latitude"],
    test["pickup_longitude"],
    test["dropoff_latitude"],
    test["dropoff_longitude"],
)

test["day_of_week"] = test["pickup_datetime"].dt.dayofweek
test["month"] = test["pickup_datetime"].dt.month
test["hour"] = test["pickup_datetime"].dt.hour

X_test = test[["distance", "day_of_week", "month", "hour"]]



In [9]:
# Predict and write submission
test["fare_amount"] = model.predict(X_test)

submission = pd.DataFrame({"key": test["key"], "fare_amount": test["fare_amount"]})

submission.to_csv("submission.csv", index=False)
print("Submission written to submission.csv, shape:", submission.shape)

Submission written to submission.csv, shape: (9914, 2)
